# Generative LLM — `career_position` Annotation

**Model:** GPT-oss 20B (LM Studio)  
**Granularity:** Broad sector codes  
**Prompting:** zero-shot (set `N_SHOTS > 0` for few-shot)  

> LM Studio must be running at `LMSTUDIO_URL` before executing inference.

## 1. Imports & config

In [11]:
from dotenv import load_dotenv
load_dotenv()

import os
import re
import time
from tqdm.auto import tqdm
from openai import OpenAI

from corex_eval import evaluate, load_inputs, load_training_data, career_position_to_sector

# --- Config ---
LMSTUDIO_URL = "http://localhost:1234/v1"
MODEL_ID     = "openai/gpt-oss-20b"   # ← verify with the "Discover models" cell below
N_SHOTS      = 0    # 0 = zero-shot; set to 3 or 5 for few-shot
SEED         = 42

## 2. Set up client & discover model IDs

In [12]:
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")

# Verify MODEL_ID against what is actually loaded in LM Studio
print("Models available in LM Studio:")
try:
    for m in client.models.list():
        print(f"  {m.id}")
except Exception as e:
    print(f"  Could not connect to LM Studio at {LMSTUDIO_URL}: {e}")

Models available in LM Studio:
  openai/gpt-oss-20b
  qwen/qwen3-next-80b
  meta/llama-3.3-70b
  text-embedding-nomic-embed-text-v1.5
  deepseek-r1-distill-qwen-7b
  devstral-small-2505
  gemma-3-27b-it-qat


## 3. Load valid codes

In [13]:
from corex_eval.config import CAREER_POSITION_SECTORS, CAREER_POSITION_SECTOR_HINTS

train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position"],
)
train_df = train_df.dropna(subset=["job_description_label", "career_position"]).reset_index(drop=True)
train_df["sector"] = train_df["career_position"].map(career_position_to_sector)

valid_codes = sorted(CAREER_POSITION_SECTORS.keys())
sector_lines = [
    f"  {k} = {v}: {CAREER_POSITION_SECTOR_HINTS[k]}"
    for k, v in sorted(CAREER_POSITION_SECTORS.items())
]
print(f"{len(valid_codes)} broad sector codes:")
for line in sector_lines:
    print(line)

/tmp/pycharm_project_967/corex_eval/silver.py:146: UserWarning: [silver] Dropped 11 row(s) for variable 'career_position' whose case_ids are not present in the gold standard.
  warnings.warn(


## 4. Build prompt & prediction parser

In [14]:
def build_system_prompt(sector_lines):
    codes_str = "\n".join(sector_lines)
    return (
        "You are an expert political scientist coding political career histories "
        "using the CoREx codebook.\n\n"
        "Task: given a career entry (job title and workplace), assign the single most "
        "appropriate broad sector code.\n"
        'Respond with ONLY the sector number (e.g. "1") — nothing else.\n\n'
        f"Broad sector codes:\n{codes_str}"
    )

def build_messages(job_description, workplace, system_prompt, shot_examples):
    user_content = f"Job title: {job_description}\nWorkplace: {workplace or 'unknown'}"
    messages = [{"role": "system", "content": system_prompt}]
    for desc, wp, sector in shot_examples:
        messages.append({"role": "user", "content": f"Job title: {desc}\nWorkplace: {wp or 'unknown'}"})
        messages.append({"role": "assistant", "content": sector})
    messages.append({"role": "user", "content": user_content})
    return messages

def parse_prediction(response_text, valid_codes):
    """Robustly map model response to a valid sector code."""
    text = response_text.strip().strip("\"'")
    if text in valid_codes:
        return text
    import re
    match = re.search(r"\b(\d+)\b", text)
    if match and match.group(1) in valid_codes:
        return match.group(1)
    return text  # no match — will be scored wrong

SYSTEM_PROMPT = build_system_prompt(sector_lines)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

System prompt: 1141 chars


## 5. Few-shot examples (skipped when `N_SHOTS=0`)

In [15]:
if N_SHOTS > 0:
    shot_pool = (
        train_df.groupby("sector", group_keys=False)
        .apply(lambda g: g.sample(1, random_state=SEED))
        .reset_index(drop=True)
    )
    shot_examples = (
        shot_pool.sample(min(N_SHOTS, len(shot_pool)), random_state=SEED)
        [["job_description_label", "workplace_label", "sector"]]
        .values.tolist()
    )
    print(f"Using {len(shot_examples)} few-shot examples")
else:
    shot_examples = []
    print("Zero-shot mode")

Zero-shot mode


## 6. Load test inputs

In [16]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df.head()

,case_id,spell_index,job_description_label,career_position,workplace_label
0,002002,1,Minister of Infrastructure and Energy,105 = Minister with portfolio (including Europ...,Government of Albania
1,002002,2,General Director,514 = Regulatory Agencies: Transport,"ALBCONTROL, JSC"
2,002002,3,Public Relations Advisor and Director of the M...,"444 = Political employment, local level, e.g. ...",Municipality of Tirana
3,002015,1,Minister of Interior,105 = Minister with portfolio (including Europ...,Government of Albania
4,002015,3,Secretary General,128 = Other political executive triangle position,Socialist Party


## 7. Run inference

In [17]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    messages = build_messages(
        row["job_description_label"],
        row.get("workplace_label", ""),
        SYSTEM_PROMPT,
        shot_examples,
    )
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=0,
            max_tokens=64,
            seed=SEED,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.1)
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

,case_id,spell_index,predicted_code
0,002002,1,1
1,002002,2,6
2,002002,3,2
3,002015,1,1
4,002015,3,4


## 8. Evaluate

In [18]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="broad",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/career/lmstudio_gptoss20b_broad/config.yaml",
)

[corex_eval] Results submitted to GitHub register.
  Repo      : Tombellens/corex_eval
  File      : results/register.csv
  Task      : annotation / career_position
  Contributor: tom
  Timestamp : 2026-03-23T15:17:38.428888+00:00

──────────────────────────────────────────────────
  CoREx evaluation — annotation / career_position
──────────────────────────────────────────────────
  Cases evaluated : 3553
  Cases skipped   : 0
  Accuracy     : 0.6085
  Macro F1     : 0.315357
  Weighted F1  : 0.611632

  Per-country accuracy:
    ALB                            0.4528  (n=53)
    AUT                            0.5417  (n=96)
    BGR                            0.4286  (n=49)
    BIH                            0.6957  (n=69)
    CZE                            0.6715  (n=137)
    DEU                            0.6678  (n=298)
    DNK                            0.7374  (n=179)
    ESP                            0.6215  (n=251)
    EST                            0.7143  (n=77)
    FRA       